# Optimización de Hiperparámetros con Keras Tuner – Hyperband


## 1. Importaciones y configuración
Usaremos MNIST para que el foco esté en el **algoritmo de optimización** y no en el dataset.


In [1]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)
print("GPUs disponibles:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPUs disponibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



## 2. Carga del dataset


In [2]:

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("Shape train:", x_train.shape)


Shape train: (60000, 28, 28, 1)



## 3. Definición del espacio de búsqueda
Aquí es donde entra **Random Search**:  
Hyperband muestrea combinaciones al azar dentro de este espacio.


In [3]:

def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(28, 28, 1)))

    # Número de filtros
    filters = hp.Choice("filters", [16, 32, 64])
    model.add(layers.Conv2D(filters, 3, activation="relu"))
    model.add(layers.MaxPooling2D())

    # Dropout
    dropout = hp.Float("dropout", 0.0, 0.5, step=0.25)
    model.add(layers.Dropout(dropout))

    model.add(layers.Flatten())

    # Unidades densas
    units = hp.Int("units", 64, 256, step=64)
    model.add(layers.Dense(units, activation="relu"))

    model.add(layers.Dense(10, activation="softmax"))

    # Learning rate
    lr = hp.Float("lr", 1e-4, 1e-2, sampling="log")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model



## 4. Configuración de Hyperband
Aquí entra **Successive Halving**:
- Muchos modelos entrenan pocas epochs
- Solo los mejores reciben más presupuesto


In [4]:

tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=20,
    factor=3,
    directory="hyperband_demo",
    project_name="mnist"
)


I0000 00:00:1768849506.967873   60013 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13794 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:07:00.0, compute capability: 8.9



## 5. Ejecución de la búsqueda


In [5]:

tuner.search(
    x_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    verbose=1
)


Trial 30 Complete [00h 00m 23s]
val_accuracy: 0.9862499833106995

Best val_accuracy So Far: 0.9898333549499512
Total elapsed time: 00h 05m 40s



## 6. Mejores hiperparámetros encontrados


In [6]:

best_hp = tuner.get_best_hyperparameters(1)[0]

for k in best_hp.values:
    print(f"{k}: {best_hp.get(k)}")


filters: 16
dropout: 0.5
units: 192
lr: 0.0013176799132117777
tuner/epochs: 20
tuner/initial_epoch: 7
tuner/bracket: 2
tuner/round: 2
tuner/trial_id: 0014



## 7. Comparativa entre configuraciones
Vamos a comparar **las mejores 5 configuraciones** para justificar
por qué la ganadora es realmente superior.


In [7]:

results = []

for trial in tuner.oracle.trials.values():
    results.append({
        "trial_id": trial.trial_id,
        "val_accuracy": trial.metrics.get_best_value("val_accuracy"),
        **trial.hyperparameters.values
    })

df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
df.head()


,trial_id,val_accuracy,filters,dropout,units,lr,tuner/epochs,tuner/initial_epoch,tuner/bracket,tuner/round,tuner/trial_id
16,0016,0.989833,16,0.50,192,0.001318,20,7,2,2,0014
24,0024,0.989750,32,0.50,256,0.003467,20,7,1,1,0019
26,0026,0.988333,32,0.50,192,0.005134,20,0,0,0,NaN
25,0025,0.988000,16,0.25,192,0.001174,20,7,1,1,0020
19,0019,0.987750,32,0.50,256,0.003467,7,0,1,0,NaN


In [8]:
best_models = tuner.get_best_models(num_models=1)
best_model = best_models[0]

/home/gibran/anaconda3/envs/tf219/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [9]:
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2704)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 192)            │       519,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,930 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 521,450 (1.99 MB)

 Trainable params: 521,450 (1.99 MB)

 Non-trainable params: 0 (0.00 B)


## 8. Análisis crítico (discusión)
Observa:
- Qué hiperparámetros aparecen en los mejores trials
- Si configuraciones similares tienen rendimiento parecido
- Cómo Hyperband descartó temprano configuraciones malas

**Conclusión clave**:
> El mejor modelo no es el más complejo, sino el que equilibra
capacidad, regularización y tasa de aprendizaje.



## 9. Conclusiones finales

- Hyperband = Random Search + Successive Halving
- Reduce drásticamente el costo computacional
- Usa la GPU automáticamente vía TensorFlow
- No paraleliza trials por defecto

**Pregunta para el alumno**:
¿Crees que esta estrategia funcionaría igual de bien en Reinforcement Learning?
